# TTI Trace Collection And Analysis

## Preparation of Jupyter Notebook

### Reading testline configuration

The below code is used, to determine the current testline configuration by the host name.

With the host name information, the testline configuration is loaded, so you can later on
work with the common keywords on the testline.

You need to execute the cell (by pressing [Ctrl]+[Enter] on the cell or pressing the play button)

After running the cell, an ip address is printed. If the ip address corresponds to the address of the local machine,
the correct testline configuration is loaded.

**This Cell must be ALWAYS called first**

In [ ]:
*** Settings ***
Library    jupyter_helper.py
Resource    ../ute_workarounds.robot
*** Test Cases ***
Load TL Config
    Load Local Testline Config
    UTE Cloud Workaround
    Log    HostPC is ${tl.test_pc}
    ${host_ip}=    Get Host IP
    ${local_ip}=   Set Variable   ${tl.test_pc.default_ssh_connection_details.addr}
    Should Be Equal    ${host_ip}   ${local_ip}    The tl config IP ${local_ip} does not fit the host ip ${host_ip}
    
    Set Variable   ${local_ip}
    

After having the testline configuration loaded with the first cell on this page, you can start interacting with your testline. 

The next cell is using the keywords from ute_workarounds to bring the testline in the OnAir status

After executing this Keyword, you are able to proceed with further interactions, like attaching UES, collecting logs, ...

*** Settings ***
Resource    ../ute_workarounds.robot

*** Test Cases ***
Bring Airphone to on Air status
    UTE Cloud Workaround
    ${wts_status}=      Run Keyword and Return Status      Bring Airphone TL to on Air status without reboot
    Run Keyword If      ${wts_status} == False             Bring Airphone TL to on Air status

### Test Setup and Teardown to connect to running egate and edaemon

Since Jupyter is instanciating the ute_wtssim again again, with each start of a new cell, the alias from a previous run cell is no more valid. A workaround for this is to define a test setup and a test teardown to setup the utw_wtssim with each test case, and tear it down after each test case.

In [ ]:
*** Settings ***
Library     ute_wtssim

Test Setup      Connect to running egate
Test Teardown   Teardown the wts sim
    

*** Keywords ***
Connect to running egate
    ${wts_alias}=      Set Variable    ap_egate
    ${egate_port}=     Set Variable    ${tl.core.egate_port}
    ${edaemon_alias}=  Set Variable    ap_edaemon
    ${edaemon_port}=   Set Variable    ${tl.core.edaemon_port}
    Set Suite Variable    ${wts_alias}
    Set Suite Variable    ${edaemon_alias}

    ute_wtssim.setup_egate    egate_port=${egate_port}  host_ip=${tl.core.default_ssh_connection_details.addr}
    ...    tool_dir=${tl.core.dir_path}  alias=${wts_alias}
    ute_wtssim.setup_edaemon    edaemon_port=${edaemon_port}  host_ip=${tl.core.default_ssh_connection_details.addr}
    ...    tool_dir=${tl.core.dir_path}  alias=${edaemon_alias}
    ${wts_status}=    Run Keyword and Return Status
    ...    Send Egate Command and verify output    cmd=sim_info    pass_regexp=Process ID    wts_alias=${wts_alias}
    ${sim_status}=    Run Keyword and Return Status
    ...    Send Egate Command and verify output    cmd=sim_state_disp    pass_regexp=The state is ACTIVE    wts_alias=${wts_alias}
    
    Run Keyword If    $wts_status == False or $sim_status == False    Fail   Unable To Connect to running Egate

Teardown the wts sim
    ute_wtssim.teardown_egate     alias=${wts_alias}
    ute_wtssim.teardown_edaemon   alias=${edaemon_alias}



### Helper functions to log/output to the Jupyter Notebook

The following code provides an inline robot python library to provide log outputs to the Jupyter Notebook.

| Keyword | Description |
| --- | --- |
|  `Show`  | The different keywords shall be used on different types of variables. I.e. if You want to print a dataframe use the keyword. |
| `Jupyter Display JSON` | For a foldable output of a complex json structure, use the keyword |
| `Jupyter Display Text` | For any textual output of i.e. a variable content, use the keyword |
| `Jupyter Display Markdown` | For displaying a markdown formatted text. |

In [ ]:
%%python module JupyterDisplay

from IPython.display import display_pretty, Markdown, display, Code, JSON

def jupyter_display_markdown(text):
    display(Markdown(text))
    
def jupyter_display_json(output_object):
    display(JSON(output_object))

def jupyter_display_text(text):
    display(Code(text))
    
def show(text):
    display(text)


In [ ]:
%%python module AdaptPythonPath

import sys
sys.path.append("/home/ute/DEVELOPMENT/robotws")

## Collection of TTI Traces

The tti trace can produce a big amount of data. So be careful about the timespan for the TTI trace collection. Hence, take care for only collecing the TTI trace with the minimum needed duration.

To start the collection of the TTI Trace, you need to call the keyword `Start TTI Traces in Streaming mode` from the common resource file *ttitrace.robot*

To stop and save the TTI Trace, use the keyword `Stop TTI Traces in Streaming Mode`. This keyword expects the argument *tti_trace_path*, which defines the output path of where the TTI trace shall be written to. The further argument *tti_trace_filename* defines the filename. Your test case needs to remember this filename for further processing.

The following test case shows the usage of the two keywords, the tti trace is then stored in the suite variable `${tti_trace}` for further evaluation.





In [ ]:
%%python module CommissioningHelper
import json
from IPython.display import display_pretty, Markdown, display, Code, JSON


def get_remove_route_commissioning_file():
    data = [{
        "distName":"MRBTS-1/TNLSVC-1/TNL-3/IPNO-1/IPRT-1",
        "operation":"update",
        "parameters":[
           {
                "parameterName":"staticRoutes",
                "value": [],
                "operation":"delete",
            }
        ]
    }]
    filename = "/tmp/remove_route.json"
    with open(filename, 'w') as commissioning_file:
        json.dump(data, commissioning_file)
    
    return filename

def get_commissioning_parameter_file(gateway="172.16.2.101"):
    # value, dist_name, p_name
    data = [{
        "distName": "MRBTS-1/TNLSVC-1/TNL-3/IPNO-1/IPRT-1",
        "operation": "update",
        "parameters": [
           {
                "parameterName": "staticRoutes",
                "value": [
                    {
                        "destIpAddr": "10.0.1.1",
                        "destinationIpPrefixLength": "32",
                        "gateway": gateway,
                        "preference": "1"
                    }
                ],
                "operation": "update"
            }
        ]
    }]
    
    filename = "/tmp/commission.json"
    with open(filename, 'w') as commissioning_file:
        json.dump(data, commissioning_file)
    
    return filename

def route_exists(iprt_data):
    try:
        for parameter in  iprt_data['parameters']:
            if parameter['complex_parameter']['name'] == 'staticRoutes':
                return True
    except KeyError:
        return False
    return False

def get_routes(iprt_data):
    try:
        for parameter in  iprt_data['parameters']:
            if parameter['complex_parameter']['name'] == 'staticRoutes':
                return parameter['complex_parameter']['value']
    except KeyError:
        return []
    return []

    
def get_route_parameters(iprt_data):
    routes = get_routes(iprt_data)
    data = []
    for route in routes:
        data.append({item.simple_parameter.name: item.simple_parameter.value  for item in route.structure_parameters})
    return data
        

In [ ]:
*** Settings ***
Library     ../python/gnb.py   WITH NAME   gnb_lib
Library     ../ResourceCI/ttitrace_helper.py
Library     CommissioningHelper
Library     JupyterDisplay

*** Test Cases ***
Check the Gating for TTI Tracing
    Set Log Level    TRACE
    Commission Removal of Route for TTI Tracing

    Get Currently Set Static Route for TTI Tracing
    
*** Keywords ***
Commission Static Route for TTI Tracing
    ${json_delta_file}=   Get Commissioning Parameter File     gateway=172.16.2.101
    Delta Commissioning json vDU and wait for success  ${json_delta_file}    gnb_id=0    du_id=0
    
   
Get Currently Set Static Route for TTI Tracing
    [Arguments]   ${gnb_id}=0   ${du_id}=0


    ${target_dir}=   testline.Get testcase log directory

    ${vdu_scf}=           Get DU current Plan file   gnb_id=${gnb_id}   resultConfigDir=${target_dir}
    ${vdu_scf_content}=   Get File   ${vdu_scf}
    ${iprt_dist_names}=   gnb_lib.Get distNames From SCF     regex=MRBTS-\\d+/TNLSVC-\\d+/TNL-3/IPNO-\\d+/IPRT-\\d+
    ...                                                      scf_content=${vdu_scf_content}

    Show    ${iprt_dist_names}
    ${oam_alias}=         Connect To OAM DU   gnb_id=${gnb_id}   du_id=${du_id}
    ${iprt_params}=       Get Managed Object Parameters of Current Plan   dist_name=MRBTS-1/TNLSVC-1/TNL-3/IPNO-1/IPRT-1
    ...                                                                   vdu_connection=${oam_alias}
    
    ${exists}=    route_exists   ${iprt_params}
    
    ${route_params}    Get Route Parameters   ${iprt_params}
    Show   ${route_params}
    
    Disconnect from OAM DU   gnb_id=0


In [ ]:
*** Settings ***
Resource    ../wts_ue.robot
Resource    ../ttitrace.robot
Library     JupyterDisplay
Library     OperatingSystem
Library     AdaptPythonPath

*** Test Cases ***
Test Collect TTI trace from pcap to excelCsv with Some Ue Action
    Set Log Level   TRACE
    ${trace_path}=     Set Variable    ${CURDIR}/traces
    Create Directory   ${trace_path}

    Start TTI Traces in Streaming mode
    Ue Register
    Sleep   30 sec
    Ue Deregister
    ${tti_trace}=    Stop TTI Traces in Streaming Mode  tti_trace_path=${trace_path}  tti_trace_filename=ttitrace_streaming
    Set Suite Variable    ${tti_trace}
    
    ${tti_trace_filesize}   Get File Size   ${tti_trace}
    Show   TTI trace file size is ${tti_trace_filesize}
    Show   ${tti_trace}
    

In [ ]:
*** Settings ***
Resource    ../wts_ue.robot
Resource    ../ttitrace.robot
Library     JupyterDisplay
Library     OperatingSystem
Library    AdaptPythonPath

*** Test Cases ***
Test Collect TTI trace from pcap to excelCsv with Some Ue Action
    ${trace_path}=     Set Variable    ${CURDIR}/traces
    Create Directory   ${trace_path}
    
    ${tti_trace}=    Collect TTI Traces in Streaming mode     tti_trace_path=${trace_path} 
    ...                                                       tti_trace_filename=ttitrace_streaming
    
    Set Suite Variable    ${tti_trace}
    
    ${tti_trace_filesize}   Get File Size   ${tti_trace}
    Show   TTI trace file size is ${tti_trace_filesize}
    Show   ${tti_trace}
    

In [ ]:
*** Settings ***
Resource    ../wts_ue.robot
Resource    ../ttitrace.robot
Library     JupyterDisplay
Library     OperatingSystem

*** Test Cases ***
Test Collect TTI trace on all in cloud with separating Creation
    ${trace_path}=     Set Variable    ${CURDIR}/traces
    Create Directory   ${trace_path}
    
    ${collector}=    Create TTI Trace Collector
    
    Start TTI Trace Generation    ${collector}
    
    Stop TTI Trace Generation
    
    Collect TTI Trace and teardown collectors
    


In [ ]:
*** Settings ***
Library    taf.collectors.tcpdump      WITH NAME    tcpdump
Library    taf.collectors.l2stream3    WITH NAME    l2stream3
Library    taf.collectors.core         WITH NAME    core

Library    ../python/gnb.py            WITH NAME    gnb_lib

Resource    ../ute_workarounds.robot
Resource    ../wts_ue.robot
Library     ttitrace_helper.py
Library     OperatingSystem

Library     JupyterDisplay

*** Test Cases ***
Get L2stream logs using cOAM cli from vDU
    #Create Static Route for TTI Tracing

    ${webservice}=  Create Dictionary
    ...  host=192.168.255.129
    ...  user=Nemuadmin
    ...  password=nemuuser
    @{nr_cell_dns}    Create List    MRBTS-1/NRBTS-1/NRCELL-1

    ${l2_stream_config}=  Create Dictionary
    ...  trace_type=Component
    ...  target_component=L2Ps
    ...  nr_cell_dn_list=${nr_cell_dns}
    @{l2_stream_configuration}    Create List    ${l2_stream_config}

    tcpdump.create tcpdump collector
    ...  name=tcpdump_vdu
    ...  options=-i eth3 udp and dst 10.0.1.1 and dst port 5678
    ...  sudo_password=ute

    l2stream3.create vdu l2stream collector
    ...  name=l2stream
    ...  webservice=${webservice}
    ...  destination_ip_addr=10.0.1.1
    ...  collector=tcpdump_vdu
    ...  l2_stream_configuration=${l2_stream_configuration}
    core.setup_collectors      names=l2stream
    core.start_collectors      names=l2stream

    Sleep    20s

    ${target_dir}=   testline.get testcase log directory

    core.stop_collectors        names=l2stream
    core.collect_collectors     names=l2stream   target_dir=${target_dir}
    core.teardown_collectors    names=l2stream

    ${tti_trace}=  Set Variable   ${target_dir}/l2stream/tcpdump_vdu/tcpdump_vdu.pcap

    ${pcap_size}=   Get File Size   ${tti_trace}

    Show   TTI trace file size is ${pcap_size}
    Show   ${tti_trace}

# TTI Trace Conversion and Analysis
## Convert TTI Trace to Pandas Dataframe

In [ ]:
*** Settings ***
Library     AdaptPythonPath
Resource    ../ttitrace.robot
Library     JupyterDisplay


*** Test Cases ***
Convert TTI Trace to dataframe
    ${target_dir}=    Set Variable    ${CURDIR}/traces
    ${output_file}=   Set Variable    ${target_dir}/sample_ttitrace
    
    ${event_list}=    Create List     dlFdSchedData    ulFdSchedData
    
    ${tti_df}=   Convert TTI trace from pcap to data frame     input_file_path=${tti_trace}
    ...                                                        events_list=${event_list}
    ...                                                        write_to_pickle=${True}
    
    
    Show   ${tti_df[['rnti', 'physCellId']].drop_duplicates()}
    Show   ${tti_df[['dlFdSchedData.rrmDlUsedMcs']].dropna()}
    Set Suite Variable   ${tti_df}
    
    


## Python Analysis from TTI Traces

In [ ]:
%%python module AnalyseTTITrace
import sys
sys.path.append("/home/ute/DEVELOPMENT/robotws")

from testsuite.Ulm.ET.resources.tti_analysis.helper_functions import display_df_as_html

import pandas as pd


def dataframe_should_not_be_empty(tti_df):
    assert pd is not None, "Init pandas was run but Pandas is not loaded!"
    assert len(tti_df) > 0, "The given tti dataframe is empty!"


def log_dataframe(df, log_level='INFO'):
    display_df_as_html(df=df, log_level=log_level)

In [ ]:
*** Settings ***
Library   AnalyseTTITrace

*** Test Cases ***
Analyse the converted tti trace
    Dataframe should not be Empty    ${tti_df}

    Log Dataframe   ${tti_df}